[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ItunuAjiboye/Tutorial_Host_Pathogen_Protein_Protein_Interaction_Prediction/blob/main/notebooks/03_Feature_Extraction.ipynb)
> This notebook is designed to be run in Google Colab. Click on "Open in Colab" to continue.

# Machine Learning for Host–Pathogen Protein–Protein Interaction Prediction

## Overview

In this notebook, we will build and evaluate **machine-learning models for predicting host–pathogen protein–protein interactions (HPPI)**.

The goal is to use protein sequence-derived features to train classification models that can distinguish between:

* **Positive interactions** — host–pathogen protein pairs known or labelled as interacting.
* **Negative interactions** — host–pathogen protein pairs used as non-interacting examples.

The notebook takes the feature datasets generated in the previous stages of the workflow and uses them to train multiple machine-learning algorithms.

### What you will learn

By the end of this tutorial, you will understand how to:

1. Prepare feature matrices and target labels for machine learning.
2. Separate metadata from numerical ML features.
3. Scale features appropriately for models that require standardization.
4. Train different machine-learning classifiers.
5. Generate predictions for unseen test data.
6. Evaluate model performance using multiple classification metrics.
7. Generate ROC curves and performance visualizations.
8. Examine the effect of different **positive-to-negative class ratios**.
9. Compare different protein feature representations, such as:

   * **Amino Acid Composition (AAC)**
   * **PAAC + CTriad**
   
10. Produce results that can be used for subsequent model comparison and interpretation.

---

## Machine-Learning Workflow

The analysis follows this general workflow:

**Feature Dataset → Feature Preparation → Feature Scaling → Model Training → Prediction → Model Evaluation → Visualization**

We will apply this workflow to different feature representations and different class-imbalance ratios.

---

## 1. Import Required Libraries

We first import the Python libraries required for data manipulation, visualization, feature preprocessing, machine learning, and model evaluation.

---

## 2. Prepare the Feature Matrix and Target Variable

Before training a machine-learning model, we need to separate the dataset into:

* **X (features):** numerical protein-derived descriptors used by the model.
* **y (target):** the interaction label that the model is learning to predict.
Our datasets also contain information such as protein sequences, protein IDs, clusters, species, and partition identifiers. These are **metadata**, rather than features intended for direct model training Therefore, these columns are excluded from the feature matrix.
---

## 3. Feature Scaling

Not all machine-learning algorithms handle numerical features in the same way.

Some models, such as: Logistic Regression, Support Vector Machine (SVM) and Multi-Layer Perceptron (MLP) benefit from standardized features.

We therefore use **StandardScaler** to transform the features so that their scales are comparable.

---

## 4. Define the Machine-Learning Models

We will compare several machine-learning algorithms rather than relying on a single model.

The models included in this tutorial are:

1. **Logistic Regression**
2. **Support Vector Machine (SVM)**
3. **Random Forest**
4. **XGBoost**
5. **LightGBM**
6. **Multi-Layer Perceptron (MLP)**

Each algorithm approaches classification differently. Comparing multiple models allows us to examine how different learning approaches perform on the same HPPI prediction task.

---

## 5. Train the Models and Generate Predictions

#####After preparing the feature matrices, each model is trained using the training dataset. The trained model is then used to predict the interaction status of protein pairs in the test dataset.
---

## 6. Evaluate Model Performance

In this tutorial, we calculate several performance metrics:

* **Accuracy**
* **Sensitivity**
* **Specificity**
* **F1 Score**
* **Matthews Correlation Coefficient (MCC)**
* **Area Under the ROC Curve (AUC-ROC)**

Using multiple metrics gives us a broader view of classification performance.

---

## 7. Visualize Model Performance

To make the results easier to interpret, we generate several visualizations using Bar plots, ROC Curves and Heatmaps.

## 8. Investigate Class Imbalance

An important part of this HPPI machine-learning workflow is examining different ratios of positive to negative examples.

We evaluate:

* **1:1** — balanced dataset
* **1:5** — one positive interaction for every five negative examples
* **1:10** — one positive interaction for every ten negative examples

This allows us to examine how changing the class distribution affects model performance.

The datasets and output directories are configured separately for each ratio.

---

## 9. Evaluate Different Feature Representations

The same machine-learning workflow can also be applied to different protein sequence feature representations.

In this tutorial, we will evaluate feature sets such as:

* **AAC** — Amino Acid Composition
* **PAAC_CTriad** — Pseudo-Amino Acid Composition combined with Composition/Transition/Distribution of Triad features

To change the feature representation being evaluated, modify the feature_name variable. All other functions and workflow steps remain unchanged.

The workflow then loads the corresponding feature dataset and evaluates the selected models across the specified class ratios.

---

## 10. What You Should Have at the End

At the end of this notebook, you will have:

* Trained machine-learning models.
* Test-set predictions.
* Performance metrics for each model.
* ROC curves and AUC values.
* Performance comparison plots.
* Results for different positive-to-negative ratios.
* Results for different feature representations.

These outputs provide the basis for **comparing the machine-learning approaches and feature representations used for HPPI prediction**.


In [ ]:
#Mounting Google Drive to access files
import os
from google.colab import drive
drive.mount ('/content/my_drive')




In [ ]:
# =============================================================================
# MODULE: MACHINE LEARNING AND MODEL EVALUATION
# =============================================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neural_network import MLPClassifier


# =============================================================================
# FEATURE EXTRACTION
# =============================================================================

# Columns that are metadata rather than ML features must not
# be included in the machine-learning feature matrix.

non_feature_cols = [
    "host_sequence",
    "pathogen_sequence",
    "host_protein_id",
    "pathogen_protein_id",
    "host_cluster",
    "pathogen_cluster",
    "pathogen_specie",
    "host_partition",
    "pathogen_partition",
    "label",
    "_fold_id"
]


def get_X_y(df_split):
    """
    Extract feature matrix (X) and target variable (y).
    """

    feature_cols = [
        c for c in df_split.columns
        if c not in non_feature_cols
    ]

    X = df_split[feature_cols].astype(float)
    y = df_split["label"].astype(int)

    return X, y


# =============================================================================
# FEATURE SCALING
# =============================================================================

def get_X_y_scaled(train_split, other_split, scaler=None):
    """
    Scale features using parameters learned from the training data only.

    If a scaler is not supplied, a new StandardScaler is fitted on the
    training data. The same scaler is then applied to the other dataset.
    """

    X_train, y_train = get_X_y(train_split)
    X_other, y_other = get_X_y(other_split)

    if scaler is None:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
    else:
        X_train_scaled = scaler.transform(X_train)

    X_other_scaled = scaler.transform(X_other)

    X_train_scaled = pd.DataFrame(
        X_train_scaled,
        columns=X_train.columns,
        index=X_train.index
    )

    X_other_scaled = pd.DataFrame(
        X_other_scaled,
        columns=X_other.columns,
        index=X_other.index
    )

    return (
        X_train_scaled,
        y_train,
        X_other_scaled,
        y_other,
        scaler
    )


# =============================================================================
# MACHINE-LEARNING MODELS
# =============================================================================

def get_models():

    return {
        "LogisticRegression": LogisticRegression(
            max_iter=1000,
            random_state=42
        ),

        "SVM": SVC(
            probability=True,
            random_state=42
        ),

        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            random_state=42
        ),

        "XGBoost": XGBClassifier(
            eval_metric="logloss",
            random_state=42
        ),

        "LightGBM": LGBMClassifier(
            n_estimators=100,
            random_state=42
        ),

        "MLP": MLPClassifier(
            hidden_layer_sizes=(256, 128, 64, 32),
            max_iter=500,
            activation="relu",
            solver="adam",
            random_state=42
        ),
    }


# Models that require standardized features
SCALED_MODELS = [
    "SVM",
    "LogisticRegression",
    "MLP"
]


# =============================================================================
# MODEL EVALUATION
# =============================================================================

def evaluate_model(name, y_true, y_pred, y_prob):

    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Sensitivity": recall_score(y_true, y_pred),
        "Specificity": (
            tn / (tn + fp)
            if (tn + fp)
            else 0
        ),
        "F1 Score": f1_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "AUC-ROC": roc_auc_score(y_true, y_prob),
    }


# =============================================================================
# MODEL TRAINING AND PREDICTION
# =============================================================================

def fit_predict(
    model,
    name,
    X_train,
    y_train,
    X_test,
    X_train_scaled=None,
    X_test_scaled=None
):

    if name in SCALED_MODELS:

        model.fit(
            X_train_scaled,
            y_train
        )

        y_pred = model.predict(
            X_test_scaled
        )

        y_prob = model.predict_proba(
            X_test_scaled
        )[:, 1]

    else:

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(
            X_test
        )

        y_prob = model.predict_proba(
            X_test
        )[:, 1]

    return y_pred, y_prob


# =============================================================================
# TRAIN/TEST EVALUATION
# =============================================================================

def run_train_test(
    train_df,
    test_df,
    models,
    label=""
):

    X_train, y_train = get_X_y(train_df)
    X_test, y_test = get_X_y(test_df)

    X_train_scaled, y_train, X_test_scaled, y_test, _ = (
        get_X_y_scaled(
            train_df,
            test_df
        )
    )

    results = []
    roc_data = {}

    for name, model in models.items():

        y_pred, y_prob = fit_predict(
            model,
            name,
            X_train,
            y_train,
            X_test,
            X_train_scaled,
            X_test_scaled
        )

        metrics = evaluate_model(
            name,
            y_test,
            y_pred,
            y_prob
        )

        metrics["imbalance_ratio"] = label

        results.append(metrics)

        fpr, tpr, _ = roc_curve(
            y_test,
            y_prob
        )

        roc_data[name] = (
            fpr,
            tpr,
            metrics["AUC-ROC"]
        )

        print(
            f"[{label}] {name}: "
            f"AUC-ROC={metrics['AUC-ROC']:.3f}"
        )

    return pd.DataFrame(results), roc_data



# =============================================================================
# PLOTTING FUNCTIONS
# =============================================================================

def plot_grouped_bar(
    results_df,
    title,
    save_path
):

    metrics = [
        "Accuracy",
        "Sensitivity",
        "Specificity",
        "F1 Score",
        "MCC",
        "AUC-ROC"
    ]

    df_melted = results_df.melt(
        id_vars=["Model"],
        value_vars=metrics,
        var_name="Metric",
        value_name="Score"
    )

    plt.figure(figsize=(12, 6))

    sns.barplot(
        data=df_melted,
        x="Metric",
        y="Score",
        hue="Model",
        palette="Set2"
    )

    plt.title(
        title,
        fontsize=16
    )

    plt.ylabel(
        "Score"
    )

    plt.xlabel("")

    plt.ylim(
        0,
        1
    )

    plt.legend(
        title="Model",
        bbox_to_anchor=(1.05, 1),
        loc="upper left"
    )

    plt.tight_layout()

    plt.savefig( f"{save_path}.tiff", dpi=600, bbox_inches="tight", format="tiff", pil_kwargs={"compression": "tiff_lzw"} )

    plt.show()


def plot_roc(
    roc_data,
    title,
    save_path
):

    fig, ax = plt.subplots(
        figsize=(7, 7)
    )

    for name, (fpr, tpr, auc) in roc_data.items():

        ax.plot(
            fpr,
            tpr,
            label=f"{name} (AUC = {auc:.3f})"
        )

    ax.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        color="gray",
        label="Chance"
    )

    ax.set_xlabel(
        "False Positive Rate"
    )

    ax.set_ylabel(
        "True Positive Rate"
    )

    ax.set_title(
        title
    )

    ax.legend(
        loc="lower right"
    )

    plt.tight_layout()

    plt.savefig( f"{save_path}.tiff", dpi=600, bbox_inches="tight", format="tiff", pil_kwargs={"compression": "tiff_lzw"} )

    plt.show()


def plot_heatmap(
    results_df,
    title,
    save_path,
    index_col="Model"
):

    main_metrics = [
        "Accuracy",
        "Sensitivity",
        "Specificity",
        "F1 Score",
        "AUC-ROC"
    ]

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, 5),
        gridspec_kw={
            "width_ratios": [5, 1]
        }
    )

    sns.heatmap(
        results_df.set_index(index_col)[main_metrics],
        annot=True,
        fmt=".3f",
        cmap="viridis",
        vmin=0,
        vmax=1,
        ax=axes[0],
        cbar=True,
        annot_kws={
            "fontsize": 13
        }
    )

    axes[0].set_title(
        title
    )

    sns.heatmap(
        results_df.set_index(index_col)[["MCC"]],
        annot=True,
        fmt=".3f",
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        ax=axes[1],
        cbar=True,
        annot_kws={
            "fontsize": 13
        }
    )

    axes[1].set_title(
        "MCC"
    )

    axes[1].set_yticklabels([])

    plt.tight_layout()

    plt.savefig( f"{save_path}.tiff", dpi=600, bbox_inches="tight", format="tiff", pil_kwargs={"compression": "tiff_lzw"} )

    plt.show()







###DRiver: AAC

In [ ]:
# =============================================================================
# DATASET CONFIGURATION
# =============================================================================

base_hpi_path = "/content/my_drive/MyDrive/HPI"

ratio_config = {

    1: {
        "features_dir":
            f"{base_hpi_path}/Features/Balanced",

        "ml_dir":
            f"{base_hpi_path}/ML/Balanced",

        "label":
            "1:1"
    },

    5: {
        "features_dir":
            f"{base_hpi_path}/Features/R5",

        "ml_dir":
            f"{base_hpi_path}/ML/R5",

        "label":
            "1:5"
    },

    10: {
        "features_dir":
            f"{base_hpi_path}/Features/R10",

        "ml_dir":
            f"{base_hpi_path}/ML/R10",

        "label":
            "1:10"
    }
}


# =============================================================================
# SELECT FEATURE SET
# =============================================================================

# Change this value depending on the feature representation being evaluated.
#
# Examples:
#   "AAC"
#   "PAAC_CTriad"
#   "AAC_PAAC_CTriad"

feature_name = "AAC"


# =============================================================================
# RUN ANALYSIS FOR EACH IMBALANCE RATIO
# =============================================================================

all_train_test_results = {}
all_cv_results = {}
all_lopo_hd_results = {}


for ratio, cfg in ratio_config.items():

    os.makedirs(
        cfg["ml_dir"],
        exist_ok=True
    )

    os.chdir(
        cfg["ml_dir"]
    )

    print(
        f"\n{'=' * 20} "
        f"Ratio {cfg['label']} "
        f"| Features: {feature_name} "
        f"{'=' * 20}"
    )

    features_dir = cfg["features_dir"]

    # =========================================================================
    # TRAIN/TEST
    # =========================================================================

    train_df = pd.read_csv(
        f"{features_dir}/"
        f"train_{feature_name}_split_features.csv"
    )

    test_df = pd.read_csv(
        f"{features_dir}/"
        f"test_{feature_name}_split_features.csv"
    )

    models = get_models()

    results_df, roc_data = run_train_test(
        train_df,
        test_df,
        models,
        label=cfg["label"]
    )

    results_df.to_csv(
        f"{cfg['label'].replace(':', '_')}_"
        f"{feature_name}_train_test_results.csv",
        index=False
    )

    all_train_test_results[ratio] = results_df

    plot_grouped_bar(
        results_df,
        f"Model Performance — "
        f"{cfg['label']} ({feature_name})",
        f"{ratio}_{feature_name}_train_test_barplot"
    )

    plot_roc(
        roc_data,
        f"ROC Curves — "
        f"{cfg['label']} ({feature_name})",
        f"{ratio}_{feature_name}_train_test_roc"
    )

    plot_heatmap(
        results_df,
        f"Model Performance Heatmap — "
        f"{cfg['label']} ({feature_name})",
        f"{ratio}_{feature_name}_train_test_heatmap"
    )




###Driver PAAC_CTriad

In [ ]:
# =============================================================================
# DATASET CONFIGURATION
# =============================================================================

base_hpi_path = "/content/my_drive/MyDrive/HPI"

ratio_config = {

    1: {
        "features_dir":
            f"{base_hpi_path}/Features/Balanced",

        "ml_dir":
            f"{base_hpi_path}/ML/Balanced",

        "label":
            "1:1"
    },

    5: {
        "features_dir":
            f"{base_hpi_path}/Features/R5",

        "ml_dir":
            f"{base_hpi_path}/ML/R5",

        "label":
            "1:5"
    },

    10: {
        "features_dir":
            f"{base_hpi_path}/Features/R10",

        "ml_dir":
            f"{base_hpi_path}/ML/R10",

        "label":
            "1:10"
    }
}


# =============================================================================
# SELECT FEATURE SET
# =============================================================================

# Change this value depending on the feature representation being evaluated.
#
# Examples:
#   "AAC"
#   "PAAC_CTriad"
#   "AAC_PAAC_CTriad"

feature_name = "PAAC_CTriad"


# =============================================================================
# RUN ANALYSIS FOR EACH IMBALANCE RATIO
# =============================================================================

all_train_test_results = {}
all_cv_results = {}
all_lopo_hd_results = {}


for ratio, cfg in ratio_config.items():

    os.makedirs(
        cfg["ml_dir"],
        exist_ok=True
    )

    os.chdir(
        cfg["ml_dir"]
    )

    print(
        f"\n{'=' * 20} "
        f"Ratio {cfg['label']} "
        f"| Features: {feature_name} "
        f"{'=' * 20}"
    )

    features_dir = cfg["features_dir"]

    # =========================================================================
    # TRAIN/TEST
    # =========================================================================

    train_df = pd.read_csv(
        f"{features_dir}/"
        f"train_{feature_name}_split_features.csv"
    )

    test_df = pd.read_csv(
        f"{features_dir}/"
        f"test_{feature_name}_split_features.csv"
    )

    models = get_models()

    results_df, roc_data = run_train_test(
        train_df,
        test_df,
        models,
        label=cfg["label"]
    )

    results_df.to_csv(
        f"{cfg['label'].replace(':', '_')}_"
        f"{feature_name}_train_test_results.csv",
        index=False
    )

    all_train_test_results[ratio] = results_df

    plot_grouped_bar(
        results_df,
        f"Model Performance — "
        f"{cfg['label']} ({feature_name})",
        f"{ratio}_{feature_name}_train_test_barplot"
    )

    plot_roc(
        roc_data,
        f"ROC Curves — "
        f"{cfg['label']} ({feature_name})",
        f"{ratio}_{feature_name}_train_test_roc"
    )

    plot_heatmap(
        results_df,
        f"Model Performance Heatmap — "
        f"{cfg['label']} ({feature_name})",
        f"{ratio}_{feature_name}_train_test_heatmap"
    )


